In [2]:
import sqlite3
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load dataset from SQLite
db_path = Path("/Users/keerthanavenkatesan/Documents/Data Managment/CS210FinalProject/Spotify_Popularity_Prediction/data/spotify.db")
conn = sqlite3.connect(db_path)

df = pd.read_sql("SELECT * FROM tracks", conn)

print("Loaded from SQL:", df.shape)
df.head()


Loaded from SQL: (100000, 20)


,Unnamed: 0,artist_name,track_name,track_id,popularity,year,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
0,1054304,Cabas,Amor De Mis Amores,3FTqjWmL21xi4HTTOq94EQ,38,2006,alt-rock,0.725,0.553,6,-6.319,0,0.0340,0.276000,0.000007,0.1850,0.729,90.009,206653,4
1,621408,Ketil Bjørnstad,Første sang,0Pt7ESPgrdTdaxp2f29hX2,11,2023,swedish,0.277,0.164,9,-16.743,0,0.0373,0.878000,0.000181,0.3350,0.184,89.308,459733,4
2,1121669,Project 86,Evil (A Chorus Of Resistance),75Ub3ckaoTdzgH9Azeu8cY,38,2007,alt-rock,0.486,0.927,2,-4.845,0,0.0428,0.000003,0.014500,0.0952,0.377,135.540,183373,4
3,439351,Ital Tek,Open Heart,5WEPna9GWi0NkqVLAkEKNN,18,2020,dubstep,0.411,0.442,1,-12.745,0,0.0270,0.485000,0.926000,0.1910,0.172,174.019,347610,3
4,266036,I-Roy,Irie Right,6peHySxvmZaRF9YEwUsggq,18,2017,dancehall,0.748,0.660,10,-4.648,0,0.2710,0.125000,0.000000,0.0783,0.400,75.583,196179,4


In [3]:
# columns to drop for modeling
drop_cols = ["Unnamed: 0", "track_id", "track_name", "artist_name"]

df_model = df.drop(columns=drop_cols)

print("New shape after dropping:", df_model.shape)
df_model.head()


New shape after dropping: (100000, 16)


,popularity,year,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
0,38,2006,alt-rock,0.725,0.553,6,-6.319,0,0.0340,0.276000,0.000007,0.1850,0.729,90.009,206653,4
1,11,2023,swedish,0.277,0.164,9,-16.743,0,0.0373,0.878000,0.000181,0.3350,0.184,89.308,459733,4
2,38,2007,alt-rock,0.486,0.927,2,-4.845,0,0.0428,0.000003,0.014500,0.0952,0.377,135.540,183373,4
3,18,2020,dubstep,0.411,0.442,1,-12.745,0,0.0270,0.485000,0.926000,0.1910,0.172,174.019,347610,3
4,18,2017,dancehall,0.748,0.660,10,-4.648,0,0.2710,0.125000,0.000000,0.0783,0.400,75.583,196179,4


In [4]:
from sklearn.preprocessing import StandardScaler

#  One-hot encode genre 
df_encoded = pd.get_dummies(df_model, columns=["genre"], drop_first=True)

print("Shape after genre encoding:", df_encoded.shape)

# Identify numeric columns to scale 
numeric_cols = [
    "year", "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness", "liveness",
    "valence", "tempo", "duration_ms", "time_signature"
]

# Scale numeric columns 
scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

print("Shape after scaling:", df_encoded.shape)

df_encoded.head()


Shape after genre encoding: (100000, 96)
Shape after scaling: (100000, 96)


,popularity,year,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,...,genre_ska,genre_sleep,genre_songwriter,genre_soul,genre_spanish,genre_swedish,genre_tango,genre_techno,genre_trance,genre_trip-hop
0,38,-0.882306,1.014912,-0.322483,0.200296,0.468072,-1.325298,-0.464735,-0.127747,-0.687657,...,False,False,False,False,False,False,False,False,False,False
1,11,1.618440,-1.423065,-1.765623,1.044799,-1.379110,-1.325298,-0.438741,1.573543,-0.687179,...,False,False,False,False,False,True,False,False,False,False
2,38,-0.735203,-0.285705,1.065009,-0.925707,0.729272,-1.325298,-0.395418,-0.907732,-0.647873,...,False,False,False,False,False,False,False,False,False,False
3,18,1.177132,-0.693849,-0.734279,-1.207208,-0.670646,-1.325298,-0.519873,0.462901,1.854253,...,False,False,False,False,False,False,False,False,False,False
4,18,0.735824,1.140076,0.074473,1.326300,0.764181,-1.325298,1.402100,-0.554482,-0.687676,...,False,False,False,False,False,False,False,False,False,False


In [5]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# CLUSTERING FEATURES (excluding popularity + year + time_signature) 
cluster_features = [
    "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "duration_ms"
]

# Scale only the clustering features
scaler_cluster = StandardScaler()
cluster_data = scaler_cluster.fit_transform(df_model[cluster_features])

# RUN K-MEANS 
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(cluster_data)

# Add cluster ID to the encoded modeling dataset
df_encoded["cluster_id"] = cluster_labels

print("K-means clustering complete.")
print("Cluster counts:")
print(df_encoded["cluster_id"].value_counts())
df_encoded.head()


K-means clustering complete.
Cluster counts:
cluster_id
2    30555
4    24657
5    18293
3    14898
1     9822
0     1775
Name: count, dtype: int64


,popularity,year,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,...,genre_sleep,genre_songwriter,genre_soul,genre_spanish,genre_swedish,genre_tango,genre_techno,genre_trance,genre_trip-hop,cluster_id
0,38,-0.882306,1.014912,-0.322483,0.200296,0.468072,-1.325298,-0.464735,-0.127747,-0.687657,...,False,False,False,False,False,False,False,False,False,2
1,11,1.618440,-1.423065,-1.765623,1.044799,-1.379110,-1.325298,-0.438741,1.573543,-0.687179,...,False,False,False,False,True,False,False,False,False,1
2,38,-0.735203,-0.285705,1.065009,-0.925707,0.729272,-1.325298,-0.395418,-0.907732,-0.647873,...,False,False,False,False,False,False,False,False,False,4
3,18,1.177132,-0.693849,-0.734279,-1.207208,-0.670646,-1.325298,-0.519873,0.462901,1.854253,...,False,False,False,False,False,False,False,False,False,3
4,18,0.735824,1.140076,0.074473,1.326300,0.764181,-1.325298,1.402100,-0.554482,-0.687676,...,False,False,False,False,False,False,False,False,False,2


In [6]:
from pathlib import Path

output_path = Path("//Users/keerthanavenkatesan/Documents/Data Managment/CS210FinalProject/Spotify_Popularity_Prediction/data/cleaned/sp_dataset_for_modeling.csv")

df_encoded.to_csv(output_path, index=False)

print("Final modeling dataset saved to:")
print(output_path)
print("Final shape:", df_encoded.shape)


Final modeling dataset saved to:
//Users/keerthanavenkatesan/Documents/Data Managment/CS210FinalProject/Spotify_Popularity_Prediction/data/cleaned/sp_dataset_for_modeling.csv
Final shape: (100000, 97)


In [7]:
import sqlite3

db_path = Path("/Users/keerthanavenkatesan/Documents/Data Managment/CS210FinalProject/Spotify_Popularity_Prediction/data/spotify.db")
conn = sqlite3.connect(db_path)

# Read raw tracks table
df_sql = pd.read_sql("SELECT * FROM tracks", conn)

# Add cluster_id (same ordering as original df)
df_sql["cluster_id"] = df_encoded["cluster_id"]

# Replace SQL table
df_sql.to_sql("tracks", conn, if_exists="replace", index=False)

conn.close()

print("SQL table 'tracks' updated with cluster_id.")


SQL table 'tracks' updated with cluster_id.
